In [ ]:
pip install --upgrade torchao

In [ ]:
pip install -U bitsandbytes>=0.46.1

In [ ]:
from peft import LoraConfig,get_peft_model,prepare_model_for_kbit_training
from datasets import Dataset
import torch
from transformers import AutoModelForCausalLM ,AutoTokenizer,BitsAndBytesConfig
model_name="TinyLlama/TinyLlama-1.1B-Chat-v1.0"

bnb_config=BitsAndBytesConfig(
     load_in_4bit=True,
     bnb_4bit_compute_dtype=torch.float16,
     bnb_4bit_use_double_quant=True,
     bnb_4bit_quant_type="nf4"
 )
tokenizer=AutoTokenizer.from_pretrained(model_name)
model=AutoModelForCausalLM.from_pretrained(model_name,
                                           quantization_config=bnb_config,device_map="auto")

tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = tokenizer.eos_token_id

model=prepare_model_for_kbit_training(model)
from datasets import load_dataset

Dataset = load_dataset("ibm-research/finqa"))

train_dataset = dataset["train"]
val_dataset = dataset["validation"]
test_dataset = dataset["test"]

lora_config=LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj","v_proj"],
    lora_dropout=0.1
)
model =get_peft_model(model,lora_config)

def format_data(example):
  text = f"### Instruction:\n{example['instruction']}\n\n### Response:\n{example['response']}"
  return {"text":text}

train_dataset = train_dataset.map(format_data)
val_dataset = val_dataset.map(format_data)

dataset=dataset.map(format_data)

def tokenize(example):
  tokens=tokenizer(example["text"],truncation=True,padding="max_length",max_length=512)
  tokens["labels"] = tokens["input_ids"].copy()
  return tokens

train_dataset = train_dataset.map(tokenize, batched=True)
val_dataset = val_dataset.map(tokenize, batched=True)

dataset=dataset.map(tokenize,batched=True)
from transformers import Trainer,TrainingArguments

training_args=TrainingArguments(
    output_dir="/.finetuned_model",
    per_device_train_batch_size=2,
    num_train_epochs=8,
    logging_steps=20,
    fp16=True,
    learning_rate = 1e-4,
    save_strategy="epoch",
    eval_strategy="epoch"
)

trainer=Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset
)

trainer.train()
results = trainer.evaluate()

print("Validation Loss:", results["eval_loss"])


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Map:   0%|          | 0/20 [00:00<?, ? examples/s]

Map:   0%|          | 0/9 [00:00<?, ? examples/s]

Map:   0%|          | 0/29 [00:00<?, ? examples/s]

Map:   0%|          | 0/20 [00:00<?, ? examples/s]

...

/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Epoch,Training Loss,Validation Loss
1,No log,13.274314
2,12.841249,7.091490
3,12.841249,3.945133
4,4.074119,1.160989
5,4.074119,0.429249
6,0.550168,0.346013
7,0.550168,0.325594
8,0.342794,0.319468


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/pyt

Training Loss,Validation Loss,Epoch
0.342794,0.319468,8


Validation Loss: 0.3194677531719208


In [ ]:
import torch

def generate_response(model, prompt):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=150,
            temperature=0.7,
            do_sample=False,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id

        )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [ ]:
prompt = """### Instruction:
what is risk? in the following format:
Definition:
Example:
Impact:
When to worry:

### Response:
"""
print(generate_response(model, prompt))

[transformers] Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


### Instruction:
what is risk? in the following format:
Definition:
Example:
Impact:
When to worry:

### Response:
Risk is the potential negative impact that could occur due to a particular action or decision. It is the potential negative outcome that could occur if a particular action or decision is not taken. Risk is measured in terms of probability and magnitude. When to worry:
- When the probability of the risk is high and the magnitude of the risk is significant. - When the probability of the risk is low and the magnitude of the risk is not significant.
